In [17]:
import numpy as np
import pandas as pd
train_df = pd.read_csv("data/train.csv")

TARGET = "target"

X = train_df[feature_cols].copy()

# ensure numeric safety (important because you hit dtype issues earlier)
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

In [18]:
train_df["row_sum"] = X.sum(axis=1)

train_df["row_mean"] = X.mean(axis=1)

train_df["row_std"] = X.std(axis=1)

train_df["row_max"] = X.max(axis=1)

train_df["row_nonzero_ratio"] = (X != 0).mean(axis=1)

# log-sum helps with heavy-tailed distributions (very important for CLV-type targets)
train_df["row_log_sum"] = np.log1p(X).sum(axis=1)

behavioral_features = [
    "row_sum",
    "row_mean",
    "row_std",
    "row_max",
    "row_nonzero_ratio",
    "row_log_sum"
]

feature_cols_base = [
    c for c in train_df.columns
    if c not in [TARGET, ID] + behavioral_features
]

feature_cols_fe = feature_cols_base + behavioral_features

3,000+ sparse signals → 6 dense behavioral indicators

This helps models:

- generalize better
- detect “high engagement users”
- reduce sparsity noise

In [19]:
#Now we reduce redundancy using statistical structure. - stopped for now as computation took too long
# PCA
#from sklearn.decomposition import PCA
#from sklearn.preprocessing import StandardScaler

# scale first (important for PCA)
#scaler = StandardScaler(with_mean=False)  # sparse-safe
#X_scaled = scaler.fit_transform(X)

#pca = PCA(n_components=0.95, random_state=42)  # keep 95% variance
#X_pca = pca.fit_transform(X_scaled)

#print("PCA components:", X_pca.shape[1])

In [4]:
# add to dataset - stopped for now as computation took too long
#for i in range(X_pca.shape[1]):
#    train_df[f"pca_{i}"] = X_pca[:, i]

In [5]:
# SVD as alternative - stopped for now as computation took too long
#from sklearn.decomposition import TruncatedSVD

#svd = TruncatedSVD(n_components=50, random_state=42)
#X_svd = svd.fit_transform(X)

#for i in range(X_svd.shape[1]):
#    train_df[f"svd_{i}"] = X_svd[:, i]

In [20]:
# activation & noise filtering
activation_rate = (X != 0).mean()
feature_variance = X.var()

stability_df = pd.DataFrame({
    "activation_rate": activation_rate,
    "variance": feature_variance
})

In [21]:
# flag unstable features
low_activation = stability_df[stability_df["activation_rate"] < 0.001].index

very_low_variance = stability_df[stability_df["variance"] < 1e-6].index

candidate_noise_features = set(low_activation) | set(very_low_variance)

len(candidate_noise_features)

948

In [26]:
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor
import shap

kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [27]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_squared_log_error
import numpy as np

def regression_metrics(y_true, y_pred):
    y_pred = np.nan_to_num(y_pred, nan=0.0, posinf=0.0, neginf=0.0)
    y_pred = np.clip(y_pred, 0, None)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred, squared=False)
    r2 = r2_score(y_true, y_pred)
    msle = mean_squared_log_error(y_true, y_pred)

    return mae, rmse, r2, msle

In [28]:
def run_cv(model, X, y, kf, log_target=False):

    results = []

    for train_idx, val_idx in kf.split(X):

        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        if log_target:
            y_train = np.log1p(y_train)

        model.fit(X_train, y_train)

        preds = model.predict(X_val)

        if log_target:
            preds = np.expm1(preds)

        preds = np.clip(preds, 0, None)

        results.append(regression_metrics(y_val, preds))

    return pd.DataFrame(results, columns=["MAE","RMSE","R2","MSLE"])

In [29]:
X = train_df.drop(columns=[TARGET, "ID"])
y = train_df[TARGET]
y_log = np.log1p(y)

In [30]:
#fit XGBoost as primary model
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method="hist"
)

xgb_model.fit(X, y_log)

#fit LightGBM as secondary model
lgb_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

lgb_model.fit(X, y_log)

#fit randomforest as comparison
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X, y)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.118098 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 164905
[LightGBM] [Info] Number of data points in the train set: 4459, number of used features: 3447
[LightGBM] [Info] Start training from score 14.490239


RandomForestRegressor(max_depth=10, n_estimators=200, n_jobs=-1,
                      random_state=42)

In [ ]:
results = {}

models = {
    "lgb": lgb_model,
    "xgb": xgb_model,
    "rf": rf_model
}

for name, model in models.items():

    # log models for boosting
    if name in ["lgb", "xgb"]:
        results[f"{name}_base"] = run_cv(model, train_df[feature_cols_base], y, kf, log_target=True)
        results[f"{name}_fe"]   = run_cv(model, train_df[feature_cols_fe], y, kf, log_target=True)

    else:
        results[f"{name}_base"] = run_cv(model, train_df[feature_cols_base], y, kf, log_target=False)
        results[f"{name}_fe"]   = run_cv(model, train_df[feature_cols_fe], y, kf, log_target=False)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.097369 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 132354
[LightGBM] [Info] Number of data points in the train set: 3567, number of used features: 3301
[LightGBM] [Info] Start training from score 14.490748
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.100734 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 131707
[LightGBM] [Info] Number of data points in the train set: 3567, number of used features: 3288
[LightGBM] [Info] Start training from score 14.500282
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.100269 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 131683
[LightGBM] [Info] Number of data points in the

In [15]:
summary = []

for name, df in results.items():
    summary.append([
        name,
        df["MAE"].mean(),
        df["RMSE"].mean(),
        df["R2"].mean(),
        df["MSLE"].mean()
    ])

comparison_df = pd.DataFrame(summary, columns=["model","MAE","RMSE","R2","MSLE"])
comparison_df.sort_values("MSLE")

,model,MAE,RMSE,R2,MSLE
2,xgb_base,4.122883e+06,7.305020e+06,0.207692,1.956572
3,xgb_fe,4.122883e+06,7.305020e+06,0.207692,1.956572
0,lgb_base,4.193312e+06,7.323400e+06,0.203604,2.064461
1,lgb_fe,4.193312e+06,7.323400e+06,0.203604,2.064461
4,rf_base,4.658151e+06,6.895960e+06,0.293535,3.149138
5,rf_fe,4.658151e+06,6.895960e+06,0.293535,3.149138
